<a href="https://colab.research.google.com/github/nickdhollman/Python-Projects/blob/Practicing-Different-Python-Libraries/DuckDB_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DuckDB: Complete Practical Practice Guide

[DuckDB Documentation](https://duckdb.org/docs/stable/) |
[Python API](https://duckdb.org/docs/stable/clients/python/overview) |
[SQL Reference](https://duckdb.org/docs/stable/sql/introduction) |
[Parquet](https://duckdb.org/docs/stable/data/parquet/overview)

This notebook is designed as a reusable **DuckDB syntax bank + hands-on practice notebook**.

It intentionally uses the same type of public NYC Taxi data as the Polars practice notebook to compare how the same analytical tasks look in:

- **Polars**
- **DuckDB**
- eventually pandas / Spark / other libraries

## Why DuckDB?

DuckDB is an **in-process analytical database**. You can think of it as a very fast SQL engine that runs directly inside Python without needing to set up a database server.

It is especially useful for:

- querying CSV and Parquet files directly,
- local analytics,
- exploratory SQL,
- joins and aggregations,
- analytical window functions,
- lightweight ETL,
- working between SQL and pandas / Polars / Arrow.

---

## Google Colab compatibility

This notebook is built to run in **Google Colab**.

When opened in Colab it will:

1. install DuckDB and supporting packages,
2. create `/content/data`,
3. download the public practice datasets,
4. create a local DuckDB database,
5. run all examples without requiring files from your computer.

The data and local database disappear when the Colab runtime resets, which is fine for a practice notebook.

## 1. Installation and imports

In [2]:
# Google Colab / Jupyter installation
%pip install -q duckdb polars

In [3]:
import duckdb
import pandas as pd
from pathlib import Path
from urllib.request import urlretrieve
import sys

print("Python version:", sys.version.split()[0])
print("DuckDB version:", duckdb.__version__)

Python version: 3.13.15
DuckDB version: 1.5.5


## 2. Set up Colab-safe folders and download public practice data

The notebook automatically detects whether it is running in Colab.

The primary dataset is the **NYC TLC Yellow Taxi Trip Records** Parquet file.  
The secondary dataset is the **NYC taxi-zone lookup** CSV.

Because the datasets are public, they do **not** need to be committed to GitHub.

In [4]:
IN_COLAB = "google.colab" in sys.modules

BASE_DIR = Path("/content") if IN_COLAB else Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

TRIP_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
ZONE_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

TRIP_FILE = DATA_DIR / "yellow_tripdata_2024-01.parquet"
ZONE_FILE = DATA_DIR / "taxi_zone_lookup.csv"

if not TRIP_FILE.exists():
    print("Downloading Yellow Taxi data...")
    urlretrieve(TRIP_URL, TRIP_FILE)

if not ZONE_FILE.exists():
    print("Downloading taxi-zone lookup...")
    urlretrieve(ZONE_URL, ZONE_FILE)

print("Running in Colab:", IN_COLAB)
print("Trip file:", TRIP_FILE)
print("Zone file:", ZONE_FILE)

Running in Colab: True
Trip file: /content/data/yellow_tripdata_2024-01.parquet
Zone file: /content/data/taxi_zone_lookup.csv


# Part I — DuckDB Fundamentals

## 3. Connect to DuckDB

DuckDB can be:

- **in-memory**: `duckdb.connect()`
- **persistent**: `duckdb.connect("database.duckdb")`

We will use a persistent database file in the Colab/local data directory to practice creating tables and views.

In [5]:
DB_FILE = DATA_DIR / "duckdb_practice.duckdb"

con = duckdb.connect(str(DB_FILE))

print("Connected to:", DB_FILE)

Connected to: /content/data/duckdb_practice.duckdb


## 4. Run your first SQL query

`con.sql()` executes SQL and returns a DuckDB **Relation**.

In [6]:
con.sql("""
SELECT
    42 AS answer,
    'DuckDB' AS database_name
""")

┌────────┬───────────────┐
│ answer │ database_name │
│ int32  │    varchar    │
├────────┼───────────────┤
│     42 │ DuckDB        │
└────────┴───────────────┘

## 5. Return results in different formats

DuckDB works well as a bridge between SQL and Python.

In [7]:
result = con.sql("""
SELECT
    1 AS id,
    'hello' AS message
""")

result.fetchall()

[(1, 'hello')]

In [8]:
con.sql("""
SELECT
    1 AS id,
    'hello' AS message
""").df()

,id,message
0,1,hello


In [9]:
arrow_table = con.sql("""
SELECT
    1 AS id,
    'hello' AS message
""").arrow()

arrow_table

# Part II — Query Files Directly

## 6. Query Parquet directly

One of DuckDB's biggest strengths is that we do **not** need to load a Parquet file into a database table before querying it.

In [11]:
trip_path = TRIP_FILE.as_posix()

con.sql(f"""
SELECT *
FROM read_parquet('{trip_path}')
LIMIT 5
""").df()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


DuckDB also allows a Parquet file path directly in the `FROM` clause.

In [12]:
con.sql(f"""
SELECT
    tpep_pickup_datetime,
    trip_distance,
    fare_amount,
    total_amount
FROM '{trip_path}'
LIMIT 5
""").df()

,tpep_pickup_datetime,trip_distance,fare_amount,total_amount
0,2024-01-01 00:57:55,1.72,17.7,22.70
1,2024-01-01 00:03:00,1.80,10.0,18.75
2,2024-01-01 00:17:06,4.70,23.3,31.30
3,2024-01-01 00:36:38,1.40,10.0,17.00
4,2024-01-01 00:46:51,0.80,7.9,16.10


## 7. Query CSV directly

In [13]:
zone_path = ZONE_FILE.as_posix()

con.sql(f"""
SELECT *
FROM read_csv_auto('{zone_path}')
LIMIT 10
""").df()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
5,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
6,7,Queens,Astoria,Boro Zone
7,8,Queens,Astoria Park,Boro Zone
8,9,Queens,Auburndale,Boro Zone
9,10,Queens,Baisley Park,Boro Zone


## 8. Inspect schema

In [14]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{trip_path}')
""").df()

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


# Part III — Core SQL Data Skills

## 9. Select columns

In [15]:
con.sql(f"""
SELECT
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    passenger_count,
    trip_distance,
    fare_amount,
    tip_amount,
    total_amount
FROM read_parquet('{trip_path}')
LIMIT 10
""").df()

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,fare_amount,tip_amount,total_amount
0,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,17.7,0.00,22.70
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,10.0,3.75,18.75
2,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,23.3,3.00,31.30
3,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,10.0,2.00,17.00
4,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,7.9,3.20,16.10
5,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.70,29.6,6.90,41.50
6,2024-01-01 00:49:44,2024-01-01 01:15:47,2,10.82,45.7,10.00,64.95
7,2024-01-01 00:30:40,2024-01-01 00:58:40,0,3.00,25.4,0.00,30.40
8,2024-01-01 00:26:01,2024-01-01 00:54:12,1,5.44,31.0,0.00,36.00
9,2024-01-01 00:28:08,2024-01-01 00:29:16,1,0.04,3.0,0.00,8.00


## 10. Rename columns with aliases

In [16]:
con.sql(f"""
SELECT
    PULocationID AS pickup_location_id,
    DOLocationID AS dropoff_location_id,
    trip_distance
FROM read_parquet('{trip_path}')
LIMIT 10
""").df()

,pickup_location_id,dropoff_location_id,trip_distance
0,186,79,1.72
1,140,236,1.80
2,236,79,4.70
3,79,211,1.40
4,211,148,0.80
5,148,141,4.70
6,138,181,10.82
7,246,231,3.00
8,161,261,5.44
9,113,113,0.04


## 11. Create calculated columns

Calculated fields are written directly in the `SELECT` clause.

In [17]:
con.sql(f"""
SELECT
    fare_amount,
    tip_amount,
    tip_amount / NULLIF(fare_amount, 0) AS tip_to_fare_ratio,
    tpep_dropoff_datetime - tpep_pickup_datetime AS trip_duration
FROM read_parquet('{trip_path}')
LIMIT 10
""").df()

,fare_amount,tip_amount,tip_to_fare_ratio,trip_duration
0,17.7,0.00,0.000000,0 days 00:19:48
1,10.0,3.75,0.375000,0 days 00:06:36
2,23.3,3.00,0.128755,0 days 00:17:55
3,10.0,2.00,0.200000,0 days 00:08:18
4,7.9,3.20,0.405063,0 days 00:06:06
5,29.6,6.90,0.233108,0 days 00:32:23
6,45.7,10.00,0.218818,0 days 00:26:03
7,25.4,0.00,0.000000,0 days 00:28:00
8,31.0,0.00,0.000000,0 days 00:28:11
9,3.0,0.00,0.000000,0 days 00:01:08


## 12. Literals and aliases

In [18]:
con.sql(f"""
SELECT
    'yellow_taxi' AS dataset_type,
    trip_distance,
    total_amount
FROM read_parquet('{trip_path}')
LIMIT 5
""").df()

,dataset_type,trip_distance,total_amount
0,yellow_taxi,1.72,22.70
1,yellow_taxi,1.80,18.75
2,yellow_taxi,4.70,31.30
3,yellow_taxi,1.40,17.00
4,yellow_taxi,0.80,16.10


## 13. Filter rows with `WHERE`

In [19]:
con.sql(f"""
SELECT
    trip_distance,
    fare_amount,
    total_amount
FROM read_parquet('{trip_path}')
WHERE trip_distance > 10
LIMIT 10
""").df()

,trip_distance,fare_amount,total_amount
0,10.82,45.7,64.95
1,23.90,120.0,127.94
2,11.51,44.3,67.49
3,11.48,47.8,63.36
4,20.85,70.0,82.69
5,13.74,56.9,82.61
6,20.34,80.0,86.25
7,16.40,70.0,82.69
8,23.00,125.5,134.75
9,11.10,49.2,54.20


In [20]:
con.sql(f"""
SELECT
    trip_distance,
    fare_amount
FROM read_parquet('{trip_path}')
WHERE trip_distance BETWEEN 5 AND 10
  AND fare_amount > 0
LIMIT 10
""").df()

,trip_distance,fare_amount
0,5.44,31.0
1,8.20,59.0
2,5.00,21.2
3,5.88,28.9
4,5.10,28.9
5,8.89,47.8
6,5.28,31.0
7,8.89,35.2
8,5.27,38.0
9,6.60,28.2


In [21]:
con.sql(f"""
SELECT
    payment_type,
    fare_amount
FROM read_parquet('{trip_path}')
WHERE payment_type IN (1, 2)
LIMIT 10
""").df()

,payment_type,fare_amount
0,2,17.7
1,1,10.0
2,1,23.3
3,1,10.0
4,1,7.9
5,1,29.6
6,1,45.7
7,2,25.4
8,2,31.0
9,2,3.0


## 14. Sort

In [22]:
con.sql(f"""
SELECT
    trip_distance,
    fare_amount,
    tip_amount,
    total_amount
FROM read_parquet('{trip_path}')
ORDER BY total_amount DESC
LIMIT 10
""").df()

,trip_distance,fare_amount,tip_amount,total_amount
0,0.00,5000.0,0.0,5000.00
1,0.00,5000.0,0.0,5000.00
2,0.00,2500.0,0.0,2500.00
3,0.00,2500.0,0.0,2500.00
4,0.00,2500.0,0.0,2500.00
5,31.95,2221.3,0.0,2225.30
6,233.25,1616.5,0.0,1617.50
7,0.00,1000.0,0.0,1000.00
8,142.62,912.3,0.0,940.93
9,157.25,899.0,0.0,900.00


## 15. Unique values and duplicates

In [23]:
con.sql(f"""
SELECT DISTINCT payment_type
FROM read_parquet('{trip_path}')
ORDER BY payment_type
""").df()

,payment_type
0,0
1,1
2,2
3,3
4,4


In [24]:
con.sql(f"""
SELECT
    COUNT(DISTINCT PULocationID) AS unique_pickup_locations
FROM read_parquet('{trip_path}')
""").df()

,unique_pickup_locations
0,260


# Part IV — Data Cleaning

## 16. Missing values

SQL represents missing values as `NULL`.

Common tools:

- `IS NULL`
- `IS NOT NULL`
- `COALESCE()`
- `NULLIF()`

In [25]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE passenger_count IS NULL) AS missing_passenger_count
FROM read_parquet('{trip_path}')
""").df()

,total_rows,missing_passenger_count
0,2964624,140162


In [26]:
con.sql(f"""
SELECT
    passenger_count,
    COALESCE(passenger_count, 0) AS passenger_count_filled
FROM read_parquet('{trip_path}')
WHERE passenger_count IS NULL
LIMIT 10
""").df()

,passenger_count,passenger_count_filled
0,<NA>,0
1,<NA>,0
2,<NA>,0
3,<NA>,0
4,<NA>,0
5,<NA>,0
6,<NA>,0
7,<NA>,0
8,<NA>,0
9,<NA>,0


### Fill missing values with a statistic

Here the median is calculated once in a CTE and used as the replacement.

In [27]:
con.sql(f"""
WITH trips AS (
    SELECT *
    FROM read_parquet('{trip_path}')
),
stats AS (
    SELECT median(passenger_count) AS median_passenger_count
    FROM trips
)
SELECT
    passenger_count,
    COALESCE(passenger_count, median_passenger_count) AS passenger_count_filled
FROM trips
CROSS JOIN stats
LIMIT 10
""").df()

,passenger_count,passenger_count_filled
0,1,1.0
1,1,1.0
2,1,1.0
3,1,1.0
4,1,1.0
5,1,1.0
6,2,2.0
7,0,0.0
8,1,1.0
9,1,1.0


## 17. Conditional logic with `CASE WHEN`

In [28]:
con.sql(f"""
SELECT
    trip_distance,
    CASE
        WHEN trip_distance < 2 THEN 'short'
        WHEN trip_distance < 10 THEN 'medium'
        ELSE 'long'
    END AS trip_category
FROM read_parquet('{trip_path}')
LIMIT 15
""").df()

,trip_distance,trip_category
0,1.72,short
1,1.80,short
2,4.70,medium
3,1.40,short
4,0.80,short
5,4.70,medium
6,10.82,long
7,3.00,medium
8,5.44,medium
9,0.04,short


## 18. Cast data types

In [29]:
con.sql(f"""
SELECT
    passenger_count,
    CAST(passenger_count AS BIGINT) AS passenger_count_integer
FROM read_parquet('{trip_path}')
WHERE passenger_count IS NOT NULL
LIMIT 10
""").df()

,passenger_count,passenger_count_integer
0,1,1
1,1,1
2,1,1
3,1,1
4,1,1
5,1,1
6,2,2
7,0,0
8,1,1
9,1,1


In [30]:
con.sql("""
SELECT
    TRY_CAST('123' AS INTEGER) AS valid_cast,
    TRY_CAST('not_a_number' AS INTEGER) AS failed_cast
""").df()

,valid_cast,failed_cast
0,123,<NA>


# Part V — Aggregation

## 19. Basic aggregations

In [31]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    AVG(trip_distance) AS avg_trip_distance,
    median(trip_distance) AS median_trip_distance,
    AVG(fare_amount) AS avg_fare,
    SUM(total_amount) AS total_revenue
FROM read_parquet('{trip_path}')
""").df()

,row_count,avg_trip_distance,median_trip_distance,avg_fare,total_revenue
0,2964624,3.652169,1.68,18.175062,7.945638e+07


## 20. Group by

In [32]:
payment_summary = con.sql(f"""
SELECT
    payment_type,
    COUNT(*) AS trip_count,
    AVG(trip_distance) AS avg_trip_distance,
    AVG(fare_amount) AS avg_fare,
    AVG(tip_amount) AS avg_tip,
    SUM(total_amount) AS total_amount
FROM read_parquet('{trip_path}')
GROUP BY payment_type
ORDER BY payment_type
""").df()

payment_summary

,payment_type,trip_count,avg_trip_distance,avg_fare,avg_tip,total_amount
0,0,140162,11.674403,20.016194,1.545957,3.617825e+06
1,1,2319046,3.264649,18.557432,4.169671,6.553360e+07
2,2,439191,3.259126,17.866037,0.002296,1.005067e+07
3,3,19597,2.159218,6.752569,0.014560,1.715810e+05
4,4,46628,3.140549,1.334889,0.042125,8.271008e+04


## 21. `GROUP BY ALL`

DuckDB supports `GROUP BY ALL`, which automatically groups by every selected expression that is not an aggregate.

In [33]:
con.sql(f"""
SELECT
    payment_type,
    VendorID,
    COUNT(*) AS trip_count,
    AVG(total_amount) AS avg_total_amount
FROM read_parquet('{trip_path}')
GROUP BY ALL
ORDER BY payment_type, VendorID
LIMIT 20
""").df()

,payment_type,VendorID,trip_count,avg_total_amount
0,0,1,48455,24.313358
1,0,2,91447,26.543460
2,0,6,260,47.696269
3,1,1,564618,26.837251
4,1,2,1754428,28.716370
5,2,1,104722,21.563098
6,2,2,334469,23.298238
7,3,1,8120,21.099804
8,3,2,11477,0.021838
9,4,1,3817,21.195145


## 22. Conditional aggregation with `FILTER`

In [34]:
con.sql(f"""
SELECT
    COUNT(*) AS total_trips,
    COUNT(*) FILTER (WHERE trip_distance > 10) AS long_trips,
    AVG(total_amount) FILTER (WHERE payment_type = 1) AS avg_credit_card_total
FROM read_parquet('{trip_path}')
""").df()

,total_trips,long_trips,avg_credit_card_total
0,2964624,228254,28.258861


# Part VI — Date and Time Operations

## 23. Extract datetime components

In [35]:
con.sql(f"""
SELECT
    tpep_pickup_datetime,
    CAST(tpep_pickup_datetime AS DATE) AS pickup_date,
    EXTRACT(hour FROM tpep_pickup_datetime) AS pickup_hour,
    EXTRACT(dow FROM tpep_pickup_datetime) AS pickup_day_of_week,
    EXTRACT(month FROM tpep_pickup_datetime) AS pickup_month
FROM read_parquet('{trip_path}')
LIMIT 10
""").df()

,tpep_pickup_datetime,pickup_date,pickup_hour,pickup_day_of_week,pickup_month
0,2024-01-01 00:57:55,2024-01-01,0,1,1
1,2024-01-01 00:03:00,2024-01-01,0,1,1
2,2024-01-01 00:17:06,2024-01-01,0,1,1
3,2024-01-01 00:36:38,2024-01-01,0,1,1
4,2024-01-01 00:46:51,2024-01-01,0,1,1
5,2024-01-01 00:54:08,2024-01-01,0,1,1
6,2024-01-01 00:49:44,2024-01-01,0,1,1
7,2024-01-01 00:30:40,2024-01-01,0,1,1
8,2024-01-01 00:26:01,2024-01-01,0,1,1
9,2024-01-01 00:28:08,2024-01-01,0,1,1


## 24. Calculate trip duration

In [36]:
con.sql(f"""
SELECT
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    date_diff(
        'minute',
        tpep_pickup_datetime,
        tpep_dropoff_datetime
    ) AS trip_minutes
FROM read_parquet('{trip_path}')
LIMIT 10
""").df()

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_minutes
0,2024-01-01 00:57:55,2024-01-01 01:17:43,20
1,2024-01-01 00:03:00,2024-01-01 00:09:36,6
2,2024-01-01 00:17:06,2024-01-01 00:35:01,18
3,2024-01-01 00:36:38,2024-01-01 00:44:56,8
4,2024-01-01 00:46:51,2024-01-01 00:52:57,6
5,2024-01-01 00:54:08,2024-01-01 01:26:31,32
6,2024-01-01 00:49:44,2024-01-01 01:15:47,26
7,2024-01-01 00:30:40,2024-01-01 00:58:40,28
8,2024-01-01 00:26:01,2024-01-01 00:54:12,28
9,2024-01-01 00:28:08,2024-01-01 00:29:16,1


## 25. Aggregate by hour

In [37]:
hourly_summary = con.sql(f"""
SELECT
    EXTRACT(hour FROM tpep_pickup_datetime) AS pickup_hour,
    COUNT(*) AS trip_count,
    AVG(trip_distance) AS avg_distance,
    AVG(total_amount) AS avg_total_amount
FROM read_parquet('{trip_path}')
GROUP BY pickup_hour
ORDER BY pickup_hour
""").df()

hourly_summary

,pickup_hour,trip_count,avg_distance,avg_total_amount
0,0,79094,3.732850,27.770448
1,1,53627,3.127259,25.378651
2,2,37517,2.883984,24.084306
3,3,24811,3.321278,25.905998
4,4,16742,4.545094,30.990660
5,5,18764,8.638057,36.225783
6,6,41429,12.865668,29.516095
7,7,83719,6.003183,26.139717
8,8,117209,5.441060,25.199265
9,9,128970,2.998480,25.497907


## 26. Date truncation

In [38]:
con.sql(f"""
SELECT
    date_trunc('day', tpep_pickup_datetime) AS pickup_day,
    COUNT(*) AS trip_count
FROM read_parquet('{trip_path}')
GROUP BY pickup_day
ORDER BY pickup_day
LIMIT 10
""").df()

,pickup_day,trip_count
0,2002-12-31,2
1,2009-01-01,3
2,2023-12-31,10
3,2024-01-01,81013
4,2024-01-02,75519
5,2024-01-03,82427
6,2024-01-04,102901
7,2024-01-05,103178
8,2024-01-06,97117
9,2024-01-07,67543


# Part VII — Strings

## 27. String operations

In [39]:
con.sql(f"""
SELECT
    Borough,
    lower(Borough) AS borough_lower,
    Zone,
    upper(Zone) AS zone_upper,
    length(Zone) AS zone_name_length
FROM read_csv_auto('{zone_path}')
LIMIT 10
""").df()

,Borough,borough_lower,Zone,zone_upper,zone_name_length
0,EWR,ewr,Newark Airport,NEWARK AIRPORT,14
1,Queens,queens,Jamaica Bay,JAMAICA BAY,11
2,Bronx,bronx,Allerton/Pelham Gardens,ALLERTON/PELHAM GARDENS,23
3,Manhattan,manhattan,Alphabet City,ALPHABET CITY,13
4,Staten Island,staten island,Arden Heights,ARDEN HEIGHTS,13
5,Staten Island,staten island,Arrochar/Fort Wadsworth,ARROCHAR/FORT WADSWORTH,23
6,Queens,queens,Astoria,ASTORIA,7
7,Queens,queens,Astoria Park,ASTORIA PARK,12
8,Queens,queens,Auburndale,AUBURNDALE,10
9,Queens,queens,Baisley Park,BAISLEY PARK,12


In [40]:
con.sql(f"""
SELECT *
FROM read_csv_auto('{zone_path}')
WHERE Zone ILIKE '%Airport%'
""").df()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,132,Queens,JFK Airport,Airports
2,138,Queens,LaGuardia Airport,Airports


## 28. String concatenation

In [41]:
con.sql(f"""
SELECT
    Borough,
    Zone,
    Borough || ' - ' || Zone AS borough_and_zone
FROM read_csv_auto('{zone_path}')
LIMIT 10
""").df()

,Borough,Zone,borough_and_zone
0,EWR,Newark Airport,EWR - Newark Airport
1,Queens,Jamaica Bay,Queens - Jamaica Bay
2,Bronx,Allerton/Pelham Gardens,Bronx - Allerton/Pelham Gardens
3,Manhattan,Alphabet City,Manhattan - Alphabet City
4,Staten Island,Arden Heights,Staten Island - Arden Heights
5,Staten Island,Arrochar/Fort Wadsworth,Staten Island - Arrochar/Fort Wadsworth
6,Queens,Astoria,Queens - Astoria
7,Queens,Astoria Park,Queens - Astoria Park
8,Queens,Auburndale,Queens - Auburndale
9,Queens,Baisley Park,Queens - Baisley Park


# Part VIII — Joins and Combining Data

## 29. Join pickup-zone metadata

DuckDB supports standard SQL joins including:

- `INNER JOIN`
- `LEFT JOIN`
- `RIGHT JOIN`
- `FULL JOIN`
- `CROSS JOIN`

It also supports useful analytical join types such as **SEMI**, **ANTI**, and **ASOF** joins.

In [42]:
con.sql(f"""
SELECT
    t.PULocationID,
    z.Borough AS pickup_borough,
    z.Zone AS pickup_zone,
    t.trip_distance,
    t.total_amount
FROM read_parquet('{trip_path}') AS t
LEFT JOIN read_csv_auto('{zone_path}') AS z
    ON t.PULocationID = z.LocationID
LIMIT 10
""").df()

,PULocationID,pickup_borough,pickup_zone,trip_distance,total_amount
0,48,Manhattan,Clinton East,2.31,19.68
1,231,Manhattan,TriBeCa/Civic Center,2.61,20.50
2,90,Manhattan,Flatiron,1.79,19.68
3,141,Manhattan,Lenox Hill West,6.58,39.00
4,79,Manhattan,East Village,3.40,28.08
5,140,Manhattan,Lenox Hill East,2.85,19.20
6,48,Manhattan,Clinton East,16.25,124.18
7,249,Manhattan,West Village,1.35,18.00
8,164,Manhattan,Midtown South,5.72,40.68
9,90,Manhattan,Flatiron,8.40,56.54


## 30. Join pickup and drop-off zone metadata

In [44]:
con.sql(f"""
SELECT
    pu.Borough AS pickup_borough,
    pu.Zone AS pickup_zone,
    dz.Borough AS dropoff_borough,
    dz.Zone AS dropoff_zone,
    t.trip_distance,
    t.total_amount
FROM read_parquet('{trip_path}') AS t
LEFT JOIN read_csv_auto('{zone_path}') AS pu
    ON t.PULocationID = pu.LocationID
LEFT JOIN read_csv_auto('{zone_path}') AS dz
    ON t.DOLocationID = dz.LocationID
LIMIT 10
""").df()

,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone,trip_distance,total_amount
0,Manhattan,Penn Station/Madison Sq West,Manhattan,East Village,1.72,22.70
1,Manhattan,Lenox Hill East,Manhattan,Upper East Side North,1.80,18.75
2,Manhattan,Upper East Side North,Manhattan,East Village,4.70,31.30
3,Manhattan,East Village,Manhattan,SoHo,1.40,17.00
4,Manhattan,SoHo,Manhattan,Lower East Side,0.80,16.10
5,Manhattan,Lower East Side,Manhattan,Lenox Hill West,4.70,41.50
6,Queens,LaGuardia Airport,Brooklyn,Park Slope,10.82,64.95
7,Manhattan,West Chelsea/Hudson Yards,Manhattan,TriBeCa/Civic Center,3.00,30.40
8,Manhattan,Midtown Center,Manhattan,World Trade Center,5.44,36.00
9,Manhattan,Greenwich Village North,Manhattan,Greenwich Village North,0.04,8.00


## 31. Route-level aggregation

In [46]:
top_routes = con.sql(f"""
SELECT
    pu.Zone AS pickup_zone,
    dz.Zone AS dropoff_zone,
    COUNT(*) AS trip_count,
    AVG(t.trip_distance) AS avg_distance,
    AVG(t.total_amount) AS avg_total_amount
FROM read_parquet('{trip_path}') AS t
LEFT JOIN read_csv_auto('{zone_path}') AS pu
    ON t.PULocationID = pu.LocationID
LEFT JOIN read_csv_auto('{zone_path}') AS dz
    ON t.DOLocationID = dz.LocationID
GROUP BY ALL
ORDER BY trip_count DESC
LIMIT 20
""").df()

top_routes

,pickup_zone,dropoff_zone,trip_count,avg_distance,avg_total_amount
0,Upper East Side South,Upper East Side North,21883,1.058019,15.508877
1,Upper East Side North,Upper East Side South,19402,1.044600,15.878771
2,Upper East Side North,Upper East Side North,15955,0.623932,13.050416
3,Upper East Side South,Upper East Side South,14938,0.621655,13.529931
4,Midtown Center,Upper East Side South,10275,1.074143,16.735615
5,Lincoln Square East,Upper West Side South,8980,0.985276,14.691315
6,Upper East Side South,Midtown Center,8834,1.055986,16.197359
7,Midtown Center,Upper East Side North,8766,1.945183,21.588192
8,Upper West Side South,Lincoln Square East,8675,0.879890,14.330707
9,Upper West Side South,Upper West Side North,8445,0.833905,13.602470


## 32. Semi and anti joins

In [47]:
con.sql(f"""
SELECT DISTINCT t.PULocationID
FROM read_parquet('{trip_path}') AS t
SEMI JOIN read_csv_auto('{zone_path}') AS z
    ON t.PULocationID = z.LocationID
ORDER BY t.PULocationID
LIMIT 10
""").df()

,PULocationID
0,1
1,2
2,3
3,4
4,6
5,7
6,8
7,9
8,10
9,11


In [48]:
con.sql(f"""
SELECT DISTINCT t.PULocationID
FROM read_parquet('{trip_path}') AS t
ANTI JOIN read_csv_auto('{zone_path}') AS z
    ON t.PULocationID = z.LocationID
ORDER BY t.PULocationID
""").df()

,PULocationID


# Part IX — CTEs and Subqueries

## 33. Common Table Expressions (`WITH`)

CTEs make larger SQL workflows easier to read and debug.

In [49]:
con.sql(f"""
WITH valid_trips AS (
    SELECT *
    FROM read_parquet('{trip_path}')
    WHERE trip_distance > 0
      AND fare_amount > 0
      AND total_amount > 0
),
hourly AS (
    SELECT
        EXTRACT(hour FROM tpep_pickup_datetime) AS pickup_hour,
        COUNT(*) AS trip_count,
        AVG(total_amount) AS avg_total_amount
    FROM valid_trips
    GROUP BY pickup_hour
)
SELECT *
FROM hourly
ORDER BY pickup_hour
""").df()

,pickup_hour,trip_count,avg_total_amount
0,0,75241,28.581547
1,1,50481,25.905453
2,2,34961,24.604432
3,3,22942,26.716070
4,4,15276,32.482409
5,5,17494,37.560369
6,6,39415,30.154420
7,7,80858,26.553335
8,8,113484,25.545821
9,9,125600,25.905078


## 34. Scalar subqueries

In [50]:
con.sql(f"""
SELECT
    trip_distance,
    total_amount
FROM read_parquet('{trip_path}')
WHERE total_amount > (
    SELECT AVG(total_amount)
    FROM read_parquet('{trip_path}')
)
LIMIT 10
""").df()

,trip_distance,total_amount
0,4.70,31.30
1,4.70,41.50
2,10.82,64.95
3,3.00,30.40
4,5.44,36.00
5,8.20,85.09
6,2.57,32.70
7,1.70,41.50
8,23.90,127.94
9,5.88,36.40


# Part X — Window Functions

## 35. Window expressions

Window functions calculate information across related rows **without collapsing the original rows**.

Common functions:

- `AVG() OVER (...)`
- `ROW_NUMBER()`
- `RANK()`
- `DENSE_RANK()`
- `LAG()`
- `LEAD()`

In [51]:
con.sql(f"""
SELECT
    payment_type,
    fare_amount,
    AVG(fare_amount) OVER (
        PARTITION BY payment_type
    ) AS payment_type_avg_fare
FROM read_parquet('{trip_path}')
LIMIT 15
""").df()

,payment_type,fare_amount,payment_type_avg_fare
0,2,17.7,17.866037
1,1,10.0,18.557432
2,1,23.3,18.557432
3,1,10.0,18.557432
4,1,7.9,18.557432
5,1,29.6,18.557432
6,1,45.7,18.557432
7,2,25.4,17.866037
8,2,31.0,17.866037
9,2,3.0,17.866037


## 36. Difference from the group mean

In [52]:
con.sql(f"""
SELECT
    payment_type,
    fare_amount,
    fare_amount - AVG(fare_amount) OVER (
        PARTITION BY payment_type
    ) AS fare_vs_payment_mean
FROM read_parquet('{trip_path}')
LIMIT 15
""").df()

,payment_type,fare_amount,fare_vs_payment_mean
0,2,17.7,-0.166037
1,1,10.0,-8.557432
2,1,23.3,4.742568
3,1,10.0,-8.557432
4,1,7.9,-10.657432
5,1,29.6,11.042568
6,1,45.7,27.142568
7,2,25.4,7.533963
8,2,31.0,13.133963
9,2,3.0,-14.866037


## 37. Rank trips within each payment type

DuckDB's `QUALIFY` clause is especially convenient for filtering results after window functions are calculated.

In [53]:
con.sql(f"""
SELECT
    payment_type,
    total_amount,
    ROW_NUMBER() OVER (
        PARTITION BY payment_type
        ORDER BY total_amount DESC
    ) AS row_number_within_payment
FROM read_parquet('{trip_path}')
QUALIFY row_number_within_payment <= 5
ORDER BY payment_type, row_number_within_payment
""").df()

,payment_type,total_amount,row_number_within_payment
0,0,312.99,1
1,0,268.24,2
2,0,265.12,3
3,0,239.35,4
4,0,231.84,5
5,1,2500.00,1
6,1,2500.00,2
7,1,689.68,3
8,1,586.60,4
9,1,551.59,5


## 38. `LAG()` and change from the previous row

In [54]:
con.sql(f"""
WITH daily AS (
    SELECT
        CAST(tpep_pickup_datetime AS DATE) AS pickup_date,
        COUNT(*) AS trip_count
    FROM read_parquet('{trip_path}')
    GROUP BY pickup_date
)
SELECT
    pickup_date,
    trip_count,
    LAG(trip_count) OVER (
        ORDER BY pickup_date
    ) AS previous_day_trips,
    trip_count - LAG(trip_count) OVER (
        ORDER BY pickup_date
    ) AS change_from_previous_day
FROM daily
ORDER BY pickup_date
""").df()

,pickup_date,trip_count,previous_day_trips,change_from_previous_day
0,2002-12-31,2,<NA>,<NA>
1,2009-01-01,3,2,1
2,2023-12-31,10,3,7
3,2024-01-01,81013,10,81003
4,2024-01-02,75519,81013,-5494
5,2024-01-03,82427,75519,6908
6,2024-01-04,102901,82427,20474
7,2024-01-05,103178,102901,277
8,2024-01-06,97117,103178,-6061
9,2024-01-07,67543,97117,-29574


# Part XI — Reshaping

## 39. `PIVOT`

DuckDB includes native SQL syntax for pivoting data.

In [55]:
con.sql(f"""
PIVOT (
    SELECT
        EXTRACT(hour FROM tpep_pickup_datetime) AS pickup_hour,
        payment_type
    FROM read_parquet('{trip_path}')
)
ON payment_type
USING count(*)
GROUP BY pickup_hour
ORDER BY pickup_hour
""").df()

,pickup_hour,0,1,2,3,4
0,0,6771,59547,10304,638,1834
1,1,6127,39712,6162,378,1248
2,2,3844,27870,4469,308,1026
3,3,2909,17461,3381,237,823
4,4,2860,10312,2742,241,587
5,5,2018,12581,3433,206,526
6,6,4267,29412,6750,355,645
7,7,6793,64159,11363,499,905
8,8,8447,91828,15058,616,1260
9,9,5418,101808,19480,797,1467


## 40. `UNPIVOT`

In [56]:
charges = pd.DataFrame({
    "trip_id": [1, 2, 3],
    "fare_amount": [15.0, 22.0, 8.5],
    "tip_amount": [3.0, 4.5, 1.0],
    "tolls_amount": [0.0, 6.94, 0.0],
})

con.register("charges", charges)

con.sql("""
UNPIVOT charges
ON fare_amount, tip_amount, tolls_amount
INTO
    NAME charge_type
    VALUE amount
ORDER BY trip_id, charge_type
""").df()

,trip_id,charge_type,amount
0,1,fare_amount,15.00
1,1,tip_amount,3.00
2,1,tolls_amount,0.00
3,2,fare_amount,22.00
4,2,tip_amount,4.50
5,2,tolls_amount,6.94
6,3,fare_amount,8.50
7,3,tip_amount,1.00
8,3,tolls_amount,0.00


# Part XII — Tables, Views, and Persistence

## 41. Create a table from Parquet

`CREATE TABLE ... AS SELECT ...` materializes query results inside DuckDB.

In [57]:
con.sql("DROP TABLE IF EXISTS taxi_sample")

con.sql(f"""
CREATE TABLE taxi_sample AS
SELECT *
FROM read_parquet('{trip_path}')
LIMIT 100000
""")

con.sql("SELECT COUNT(*) AS rows_in_table FROM taxi_sample").df()

,rows_in_table
0,100000


## 42. Create a view over Parquet

A view lets the file remain external while presenting it like a database object.

In [58]:
con.sql("DROP VIEW IF EXISTS taxi_external")

con.sql(f"""
CREATE VIEW taxi_external AS
SELECT *
FROM read_parquet('{trip_path}')
""")

con.sql("""
SELECT
    COUNT(*) AS trip_count,
    AVG(total_amount) AS avg_total_amount
FROM taxi_external
""").df()

,trip_count,avg_total_amount
0,2964624,26.801505


## 43. Inspect database objects

In [59]:
con.sql("SHOW TABLES").df()

,name
0,charges
1,taxi_external
2,taxi_sample


## 44. Describe a table

In [60]:
con.sql("DESCRIBE taxi_sample").df()

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


# Part XIII — Python + DuckDB Interoperability

## 45. Query a pandas DataFrame directly

DuckDB can query a pandas DataFrame without saving it to a file first.

In [61]:
people_df = pd.DataFrame({
    "name": ["Ava", "Ben", "Chloe", "Diego"],
    "department": ["Analytics", "IT", "Analytics", "Finance"],
    "salary": [85000, 92000, 88000, 97000],
})

con.sql("""
SELECT
    department,
    COUNT(*) AS employee_count,
    AVG(salary) AS avg_salary
FROM people_df
GROUP BY department
ORDER BY avg_salary DESC
""").df()

,department,employee_count,avg_salary
0,Finance,1,97000.0
1,IT,1,92000.0
2,Analytics,2,86500.0


## 46. Explicitly register a DataFrame

In [62]:
con.register("people", people_df)

con.sql("""
SELECT *
FROM people
WHERE salary >= 90000
ORDER BY salary DESC
""").df()

,name,department,salary
0,Diego,Finance,97000
1,Ben,IT,92000


## 47. Convert DuckDB results to pandas

In [63]:
taxi_pandas = con.sql("""
SELECT
    trip_distance,
    fare_amount,
    tip_amount,
    total_amount
FROM taxi_sample
LIMIT 1000
""").df()

print(type(taxi_pandas))
taxi_pandas.head()

<class 'pandas.DataFrame'>


,trip_distance,fare_amount,tip_amount,total_amount
0,1.72,17.7,0.00,22.70
1,1.80,10.0,3.75,18.75
2,4.70,23.3,3.00,31.30
3,1.40,10.0,2.00,17.00
4,0.80,7.9,3.20,16.10


## 48. Convert DuckDB results to Polars

In [64]:
taxi_polars = con.sql("""
SELECT
    trip_distance,
    fare_amount,
    tip_amount,
    total_amount
FROM taxi_sample
LIMIT 1000
""").pl()

print(type(taxi_polars))
taxi_polars.head()

<class 'polars.dataframe.frame.DataFrame'>


trip_distance,fare_amount,tip_amount,total_amount
f64,f64,f64,f64
1.72,17.7,0.0,22.7
1.8,10.0,3.75,18.75
4.7,23.3,3.0,31.3
1.4,10.0,2.0,17.0
0.8,7.9,3.2,16.1


# Part XIV — DuckDB Relational API

## 49. Create a relation

The Relational API gives you a Python-style way to build DuckDB queries.

A relation is conceptually similar to a lazy query plan.

In [65]:
taxi_rel = con.read_parquet(str(TRIP_FILE))
taxi_rel

┌──────────┬──────────────────────┬───────────────────────┬─────────────────┬───────────────┬────────────┬────────────────────┬──────────────┬──────────────┬──────────────┬─────────────┬────────┬─────────┬────────────┬──────────────┬───────────────────────┬──────────────┬──────────────────────┬─────────────┐
│ VendorID │ tpep_pickup_datetime │ tpep_dropoff_datetime │ passenger_count │ trip_distance │ RatecodeID │ store_and_fwd_flag │ PULocationID │ DOLocationID │ payment_type │ fare_amount │ extra  │ mta_tax │ tip_amount │ tolls_amount │ improvement_surcharge │ total_amount │ congestion_surcharge │ Airport_fee │
│  int32   │      timestamp       │       timestamp       │      int64      │    double     │   int64    │      varchar       │    int32     │    int32     │    int64     │   double    │ double │ double  │   double   │    double    │        double         │    double    │        double        │   double    │
├──────────┼──────────────────────┼───────────────────────┼───────────

## 50. Select, filter, sort, and limit

In [66]:
rel_filtered = (
    taxi_rel
    .select("trip_distance, fare_amount, tip_amount, total_amount")
    .filter("trip_distance > 10 AND fare_amount > 0")
    .order("total_amount DESC")
    .limit(10)
)

rel_filtered.df()

,trip_distance,fare_amount,tip_amount,total_amount
0,31.95,2221.3,0.0,2225.30
1,233.25,1616.5,0.0,1617.50
2,142.62,912.3,0.0,940.93
3,157.25,899.0,0.0,900.00
4,109.75,761.1,0.0,775.48
5,119.46,739.4,0.0,771.41
6,122.47,749.2,0.0,758.89
7,120.76,744.3,0.0,753.74
8,110.46,669.4,0.0,709.21
9,111.57,678.5,0.0,696.00


## 51. Aggregate with the Relational API

In [68]:
(
    taxi_rel
    .filter("trip_distance > 0")
    .aggregate(
        "payment_type, "
        "count(*) AS trip_count, "
        "avg(trip_distance) AS avg_distance, "
        "avg(total_amount) AS avg_total",
        "payment_type"
    )
    .order("payment_type")
    .df()
)

,payment_type,trip_count,avg_distance,avg_total
0,0,117337,13.945369,26.096042
1,1,2298442,3.293914,28.074859
2,2,430608,3.324087,22.962760
3,3,15031,2.815129,9.005361
4,4,42835,3.418642,1.684406


# Part XV — Parameterized Queries

## 52. Use query parameters

Parameters are preferable to manually concatenating user-supplied values into SQL.

In [69]:
minimum_distance = 15

con.execute(
    f"""
    SELECT
        trip_distance,
        fare_amount,
        total_amount
    FROM read_parquet('{trip_path}')
    WHERE trip_distance >= ?
    ORDER BY trip_distance DESC
    LIMIT 10
    """,
    [minimum_distance],
).df()

,trip_distance,fare_amount,total_amount
0,312722.30,14.46,22.15
1,97793.92,29.71,36.31
2,82015.45,16.56,21.56
3,72975.97,12.70,20.04
4,71752.26,41.06,49.57
5,59282.45,32.02,33.52
6,59076.43,13.82,23.17
7,58298.51,12.94,18.63
8,51619.36,16.17,24.20
9,44018.64,32.75,52.43


### 53. Multiple parameters

In [70]:
min_distance = 5
max_distance = 10

con.execute(
    f"""
    SELECT
        trip_distance,
        fare_amount,
        total_amount
    FROM read_parquet('{trip_path}')
    WHERE trip_distance BETWEEN ? AND ?
    LIMIT 10
    """,
    [min_distance, max_distance],
).df()

,trip_distance,fare_amount,total_amount
0,5.44,31.0,36.00
1,8.20,59.0,85.09
2,5.00,21.2,25.45
3,5.88,28.9,36.40
4,5.10,28.9,33.90
5,8.89,47.8,60.72
6,5.28,31.0,43.20
7,8.89,35.2,64.32
8,5.27,38.0,51.60
9,6.60,28.2,33.20


# Part XVI — Reading and Writing Files

## 54. Write query results to Parquet

In [71]:
PAYMENT_PARQUET = OUTPUT_DIR / "payment_summary.parquet"

con.sql(f"""
COPY (
    SELECT
        payment_type,
        COUNT(*) AS trip_count,
        AVG(trip_distance) AS avg_trip_distance,
        AVG(total_amount) AS avg_total_amount
    FROM read_parquet('{trip_path}')
    GROUP BY payment_type
    ORDER BY payment_type
)
TO '{PAYMENT_PARQUET.as_posix()}'
(FORMAT PARQUET)
""")

print(PAYMENT_PARQUET)

/content/output/payment_summary.parquet


## 55. Write query results to CSV

In [72]:
PAYMENT_CSV = OUTPUT_DIR / "payment_summary.csv"

con.sql(f"""
COPY (
    SELECT
        payment_type,
        COUNT(*) AS trip_count,
        AVG(trip_distance) AS avg_trip_distance,
        AVG(total_amount) AS avg_total_amount
    FROM read_parquet('{trip_path}')
    GROUP BY payment_type
    ORDER BY payment_type
)
TO '{PAYMENT_CSV.as_posix()}'
(HEADER, DELIMITER ',')
""")

print(PAYMENT_CSV)

/content/output/payment_summary.csv


## 56. Read the exported Parquet file back

In [73]:
con.sql(f"""
SELECT *
FROM read_parquet('{PAYMENT_PARQUET.as_posix()}')
""").df()

,payment_type,trip_count,avg_trip_distance,avg_total_amount
0,0,140162,11.674403,25.811737
1,1,2319046,3.264649,28.258861
2,2,439191,3.259126,22.884506
3,3,19597,2.159218,8.755475
4,4,46628,3.140549,1.773829


# Part XVII — Query Planning and Performance

## 57. `EXPLAIN`

`EXPLAIN` shows DuckDB's query plan.

In [74]:
plan = con.sql(f"""
EXPLAIN
SELECT
    payment_type,
    COUNT(*) AS trip_count,
    AVG(total_amount) AS avg_total_amount
FROM read_parquet('{trip_path}')
WHERE trip_distance > 10
GROUP BY payment_type
""").fetchall()

print("\n".join(str(row[1]) for row in plan))

┌───────────────────────────┐
│       HASH_GROUP_BY       │
│    ────────────────────   │
│         Groups: #0        │
│                           │
│        Aggregates:        │
│        count_star()       │
│          avg(#1)          │
│                           │
│       ~537,394 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│        payment_type       │
│        total_amount       │
│                           │
│       ~592,924 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│        READ_PARQUET       │
│    ────────────────────   │
│         Function:         │
│        READ_PARQUET       │
│                           │
│        Projections:       │
│        payment_type       │
│        total_amount       │
│                           │
│          Filters:         │
│     trip_distance>10.0    │
│                           │
│       ~592,924 rows       │
└─────────

## 58. `EXPLAIN ANALYZE`

`EXPLAIN ANALYZE` executes the query and reports runtime information.

In [75]:
plan = con.sql(f"""
EXPLAIN ANALYZE
SELECT
    payment_type,
    COUNT(*) AS trip_count,
    AVG(total_amount) AS avg_total_amount
FROM read_parquet('{trip_path}')
WHERE trip_distance > 10
GROUP BY payment_type
""").fetchall()

print("\n".join(str(row[1]) for row in plan))

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
 EXPLAIN ANALYZE SELECT     payment_type,     COUNT(*) AS trip_count,     AVG(total_amount) AS avg_total_amount FROM read_parquet('/content/data/yellow_tripdata_2024-01.parquet') WHERE trip_distance > 10 GROUP BY payment_type 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0831s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│                           │
│           0 rows          │
│           0.00s           │
└─────────────┬─────────────┘
┌─────────────┴───

## 59. Projection and filter pushdown

With Parquet, DuckDB can often avoid reading columns and row groups that are not needed.

That makes this pattern very powerful:

```sql
SELECT needed_columns
FROM read_parquet(...)
WHERE restrictive_filter
```

In [76]:
con.sql(f"""
SELECT
    trip_distance,
    total_amount
FROM read_parquet('{trip_path}')
WHERE trip_distance >= 20
LIMIT 20
""").df()

,trip_distance,total_amount
0,23.90,127.94
1,20.85,82.69
2,20.34,86.25
3,23.00,134.75
4,20.59,103.36
5,22.63,114.96
6,20.01,103.72
7,20.63,96.23
8,21.96,90.55
9,21.06,90.49


# Part XVIII — Practical Data Quality Pipeline

## 60. Build a cleaned taxi table

This combines:

- direct Parquet scanning,
- validation filters,
- derived columns,
- datetime calculations,
- conditional logic,
- table materialization.

In [77]:
con.sql("DROP TABLE IF EXISTS clean_trips")

con.sql(f"""
CREATE TABLE clean_trips AS
SELECT
    VendorID,
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    passenger_count,
    trip_distance,
    PULocationID,
    DOLocationID,
    payment_type,
    fare_amount,
    tip_amount,
    tolls_amount,
    total_amount,

    date_diff(
        'minute',
        tpep_pickup_datetime,
        tpep_dropoff_datetime
    ) AS trip_minutes,

    EXTRACT(hour FROM tpep_pickup_datetime) AS pickup_hour,

    CASE
        WHEN trip_distance < 2 THEN 'short'
        WHEN trip_distance < 10 THEN 'medium'
        ELSE 'long'
    END AS trip_category,

    tip_amount / NULLIF(fare_amount, 0) AS tip_to_fare_ratio

FROM read_parquet('{trip_path}')

WHERE trip_distance > 0
  AND fare_amount > 0
  AND total_amount > 0
  AND tpep_dropoff_datetime >= tpep_pickup_datetime
""")

con.sql("""
SELECT
    COUNT(*) AS clean_rows,
    AVG(trip_distance) AS avg_distance,
    AVG(trip_minutes) AS avg_trip_minutes
FROM clean_trips
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clean_rows,avg_distance,avg_trip_minutes
0,2869658,3.733001,15.755742


## 61. Build an analytics summary

In [78]:
con.sql("""
SELECT
    trip_category,
    payment_type,
    COUNT(*) AS trip_count,
    AVG(trip_distance) AS avg_distance,
    AVG(trip_minutes) AS avg_trip_minutes,
    AVG(total_amount) AS avg_total_amount
FROM clean_trips
GROUP BY ALL
ORDER BY trip_category, payment_type
""").df()

,trip_category,payment_type,trip_count,avg_distance,avg_trip_minutes,avg_total_amount
0,long,0,5998,219.445452,37.689563,72.516594
1,long,1,180957,16.249578,41.865697,87.945524
2,long,2,36325,16.289977,43.009250,76.727710
3,long,3,780,16.468744,40.800000,75.009590
4,long,4,2186,16.771825,39.623513,79.784629
5,medium,0,61966,4.136554,19.328325,29.287848
6,medium,1,797257,3.917527,20.325011,32.419913
7,medium,2,132592,4.067445,21.086046,27.707488
8,medium,3,2736,4.160318,18.830409,27.298414
9,medium,4,6661,4.303684,18.937247,28.795475


# Part XIX — Useful DuckDB-Specific SQL

## 62. `SELECT * EXCLUDE`

Keep all columns except selected ones.

In [79]:
con.sql("""
SELECT * EXCLUDE (VendorID, store_and_fwd_flag)
FROM taxi_sample
LIMIT 5
""").df()

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


## 63. `SELECT * REPLACE`

Modify selected columns without listing every other column.

In [80]:
con.sql("""
SELECT * REPLACE (
    round(total_amount, 2) AS total_amount,
    round(fare_amount, 2) AS fare_amount
)
FROM taxi_sample
LIMIT 5
""").df()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


## 64. `COLUMNS()` expressions

Select groups of columns using a regular-expression pattern.

In [81]:
con.sql("""
SELECT COLUMNS('.*amount.*')
FROM taxi_sample
LIMIT 5
""").df()

,fare_amount,tip_amount,tolls_amount,total_amount
0,17.7,0.00,0.0,22.70
1,10.0,3.75,0.0,18.75
2,23.3,3.00,0.0,31.30
3,10.0,2.00,0.0,17.00
4,7.9,3.20,0.0,16.10


## 65. Generate rows with `range()`

In [82]:
con.sql("""
SELECT *
FROM range(1, 11) AS t(number)
""").df()

,number
0,1
1,2
2,3
3,4
4,5
5,6
6,7
7,8
8,9
9,10


## 66. Lists and `UNNEST`

In [83]:
con.sql("""
SELECT
    unnest([10, 20, 30, 40]) AS value
""").df()

,value
0,10
1,20
2,30
3,40


# Part XX — Performance-Minded DuckDB Patterns

## Query Parquet directly when appropriate

For exploratory analytics, you often do **not** need to import the file into a database table first.

```sql
SELECT ...
FROM read_parquet('file.parquet')
WHERE ...
```

## Select only what you need

Prefer:

```sql
SELECT trip_distance, total_amount
FROM ...
```

instead of `SELECT *` when working with large datasets.

## Filter early

Use restrictive `WHERE` conditions as early as logically possible.

## Prefer SQL operations over Python row loops

Let DuckDB execute filtering, joins, aggregation, and window functions using its vectorized engine.

## Prefer Parquet for analytical data

Parquet is columnar and works especially well with DuckDB.

## Materialize intentionally

Use a **view** when you want to keep data external.

Use a **table** when repeated analysis makes materialization useful.

## Inspect expensive queries

Use:

```sql
EXPLAIN ...
```

and:

```sql
EXPLAIN ANALYZE ...
```

# Part XXI — DuckDB vs Polars vs pandas Mental Model

| Task | pandas | Polars | DuckDB |
|---|---|---|---|
| Main interface | Python DataFrame | Expression/DataFrame API | SQL |
| Read CSV | `pd.read_csv()` | `pl.read_csv()` | `read_csv_auto()` |
| Read Parquet | `pd.read_parquet()` | `pl.read_parquet()` | `read_parquet()` |
| Lazy execution | Limited | `scan_*()` | Query plans / relations |
| Select | `df[[...]]` | `select()` | `SELECT` |
| Filter | Boolean mask | `filter()` | `WHERE` |
| New column | assignment | `with_columns()` | expression `AS alias` |
| Group | `groupby()` | `group_by()` | `GROUP BY` |
| Join | `merge()` | `join()` | `JOIN` |
| Sort | `sort_values()` | `sort()` | `ORDER BY` |
| Window | `transform()` / rolling | `.over()` | `OVER (...)` |
| Query Parquet directly | Not SQL-native | Lazy scan | Yes |
| Persistent tables | No | No | Yes |
| SQL interface | External | `SQLContext` | Native |
| In-process analytical DB | No | No | Yes |

### Mental model

- **pandas** = Python-first DataFrame analysis
- **Polars** = expression-first DataFrame engine
- **DuckDB** = SQL-first in-process analytical database

These tools are often **complementary**, not mutually exclusive.

# Part XXII — Key DuckDB Syntax Bank

```python
import duckdb

# In-memory database
con = duckdb.connect()

# Persistent database
con = duckdb.connect("my_database.duckdb")

# SQL
con.sql("SELECT ...")

# Parameterized SQL
con.execute("SELECT ... WHERE x = ?", [value])

# Convert result
con.sql("SELECT ...").df()
con.sql("SELECT ...").arrow()
con.sql("SELECT ...").pl()
con.sql("SELECT ...").fetchall()

# Query files through Python
con.read_parquet("file.parquet")
con.read_csv("file.csv")

# Register a Python object
con.register("my_table", pandas_df)
```

```sql
-- Query Parquet
SELECT *
FROM read_parquet('file.parquet');

-- Query CSV
SELECT *
FROM read_csv_auto('file.csv');

-- Select
SELECT column_a, column_b
FROM data;

-- Filter
SELECT *
FROM data
WHERE column_a > 0;

-- New column
SELECT
    a,
    b,
    a / NULLIF(b, 0) AS ratio
FROM data;

-- Conditional logic
CASE
    WHEN x < 10 THEN 'low'
    WHEN x < 20 THEN 'medium'
    ELSE 'high'
END

-- Missing values
COALESCE(column_name, replacement_value)

-- Grouping
SELECT
    category,
    COUNT(*) AS n,
    AVG(value) AS avg_value
FROM data
GROUP BY category;

-- DuckDB convenience
GROUP BY ALL

-- Join
SELECT *
FROM left_table AS l
LEFT JOIN right_table AS r
    ON l.id = r.id;

-- CTE
WITH cleaned AS (
    SELECT *
    FROM data
    WHERE value > 0
)
SELECT *
FROM cleaned;

-- Window
AVG(value) OVER (PARTITION BY category)

ROW_NUMBER() OVER (
    PARTITION BY category
    ORDER BY value DESC
)

-- Filter window output
QUALIFY row_number() OVER (...) <= 5;

-- Date/time
EXTRACT(hour FROM timestamp_column)
date_trunc('day', timestamp_column)
date_diff('minute', start_time, end_time)

-- Create table
CREATE TABLE table_name AS
SELECT ...;

-- Create view
CREATE VIEW view_name AS
SELECT ...;

-- Export
COPY (
    SELECT ...
)
TO 'output.parquet'
(FORMAT PARQUET);

-- Query plan
EXPLAIN SELECT ...;

-- Runtime plan
EXPLAIN ANALYZE SELECT ...;
```

# Part XXIII — Optional Colab ↔ GitHub Workflow

Because this notebook is Colab-ready, a simple workflow is:

1. Upload `DuckDB_Practice.ipynb` to your GitHub branch.
2. Open the notebook from GitHub in Google Colab.
3. Practice and edit in Colab.
4. Use **File → Save a copy in GitHub** when you want to push your updated notebook back.

The current practice branch is:

`Practicing-Different-Python-Libraries`

A useful repository layout would be:

```text
Python-Projects/
│
├── Polars_Practice.ipynb
├── DuckDB_Practice.ipynb
│
└── README.md
```

Later, I will add notebooks such as:

```text
Pandas_Practice.ipynb
PySpark_Practice.ipynb
Apache_Beam_Practice.ipynb
dbt_Practice/
```

# Final Takeaway

A useful DuckDB workflow is:

```text
CSV / Parquet / pandas / Polars
            ↓
          DuckDB
            ↓
      SQL transformations
            ↓
 pandas / Polars / Arrow / Parquet
```

The most important ideas to retain are:

1. **Query files directly.**
2. **Use SQL for set-based transformations rather than Python loops.**
3. **Use Parquet filter and projection pushdown.**
4. **Learn CTEs and window functions well.**
5. **Use views for external data and tables when materialization helps.**
6. **Use `EXPLAIN` and `EXPLAIN ANALYZE` to understand performance.**
7. **Treat DuckDB as complementary to Polars and pandas.**